#PDF Loader

In [1]:

# 1. INSTALL (run once)
!pip install -U langchain langchain-community langchain-core langchain-groq langchain-huggingface faiss-cpu pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.5.1
    Uninstalling sentence-transformers-5.5.1:
      Successfully uninstalled sentence-transformers-5.5.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.6
    Uninstalling langchain-1.3.6:
      Successfully uninstalled langchain-1.3.6


In [2]:
# 2. IMPORTS
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_groq import ChatGroq

/tmp/ipykernel_12486/39755542.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
# 3. API KEY (GROQ)
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [4]:
# =========================
# 4. CREATE RAG PIPELINE
# =========================
def create_policy_qa_chain(pdf_path: str):

    # ---- Load PDF ----
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()

    # ---- Split text ----
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )
    chunks = splitter.split_documents(docs)

    # ---- Embeddings (UPDATED) ----
    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5"
    )

    # ---- Vector DB ----
    vectorstore = FAISS.from_documents(chunks, embeddings)

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": 5}
    )

    # ---- LLM (Groq) ----
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0
    )

    # ---- Prompt ----
    prompt = ChatPromptTemplate.from_template("""
You are a precise assistant for document Q&A.

Use ONLY the context below.

If answer is not present, say:
"I cannot find this information in the document."

Context:
{context}

Question:
{question}
""")

    # ---- format docs ----
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    # ---- RAG Chain ----
    rag_chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

In [8]:
from google.colab import files
files.upload()

Saving Latency variation.pdf to Latency variation.pdf


{'Latency variation.pdf': b'%PDF-1.4\n%\xd3\xeb\xe9\xe1\n1 0 obj\n<</Title (Latency variation)\n/Producer (Skia/PDF m151 Google Docs Renderer)>>\nendobj\n3 0 obj\n<</ca 1\n/BM /Normal>>\nendobj\n6 0 obj\n<</N 3\n/Filter /FlateDecode\n/Length 294>> stream\nx\x9c\x95\x90\xbdJ\xc3`\x14\x86\x9f\xd4\x82(\x8a\x83\x0e\x1d\x1c28\xb8h\x93\xa6mRpi#\x16\xd7V!\xa9S\x92\xfe \xb6IHS\xf4\x02tspu+.\xde\x80\xe8e(\x08\x0e\xe2\xe0%\x88\xa0\xb3|\r\x92:t\xf0\xc0\x81\x87\xf7\x1c\xce\xcf\x0b\x99\x1c@V\x81\x81\x1fG\x8dzM\xb6\xec\x96<\xff\x8e\x84$*\xe0x\xc3\x90\xd9!\xc1\xf7K\xd2\xfb\xbc\xc5\xffc\xa1\xdd\x19z\xc0\x07\x10G\x96\xdd\x02\xa9\r\xac\xf5\x12>\x13\xec&|)\xf84\x0ec\x90\xc6\x82\xa3\x83\x86\t\xd2\x1d\xb0\xd9\x9bbw\x8a\xbd0\x12\xfdo\xc0\xce\xa0?\xf2\xd2\xbbY\xea\xf8\x87M\xc0\x02\xd6\xa9\x13\x10\xd0\xa3O\x87<MN8\xc6!\x8f\x8eI\x89=\xaa\x14P)\xa1\xa2QA\xa78\xc9*\nE\x0c\xca\xd4\xa8abb\xa0\xa1\xa3\xa1\xb1K\x89\x8a\xf03Y\x19\xdc\x80\xf1\x05sW\xa9\xe6^\xc3\xc3\x05\xe4^Smc\x0c+\xe7p\xff\x98j\xa9\xc7\xa1\x139\x13)\

In [10]:
import os
os.listdir("/content")

['.config', 'Latency variation.pdf', '.ipynb_checkpoints', 'sample_data']

In [11]:
# 5. LOAD YOUR PDF
pdf_file = "/content/Latency variation.pdf"
qa_chain = create_policy_qa_chain(pdf_file)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
# 6. TEST QUERIES
questions = [
    "What is the document about?",
    "What are descriptive statistics in the document?",
    "What is the total strength?"
]

for q in questions:
    print("\nQ:", q)
    print("A:", qa_chain.invoke(q))


Q: What is the document about?
A: The document appears to be about analyzing the latency and retrieval performance of a system in responding to various queries (Q1-Q8), and how factors such as reasoning complexity, token length, and retrieval quality impact the system's performance.

Q: What are descriptive statistics in the document?
A: I cannot find this information in the document.

Q: What is the total strength?
A: I cannot find this information in the document.
